# 06 - Cox Proportional Hazards

Core baseline dễ giải thích.

**Event = resolution**, nên:
- Hazard Ratio > 1 → resolution hazard cao hơn → xu hướng resolve nhanh hơn.
- Hazard Ratio < 1 → resolution hazard thấp hơn → xu hướng tồn đọng lâu hơn.

Không gọi Hazard Ratio là "nguyên nhân".

In [1]:
from pathlib import Path
import sys, yaml, pandas as pd, numpy as np

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

with open(ROOT / "configs" / "experiment.yaml", "r", encoding="utf-8") as f:
    CFG = yaml.safe_load(f)

print("ROOT =", ROOT)
print("random_seed =", CFG["random_seed"])

ROOT = d:\VNUK\Eureka 2026\Eureka_2026
random_seed = 42


In [2]:
from sklearn.model_selection import GroupShuffleSplit

MODE = "day0"  # đổi thành "day7" nếu muốn chạy landmark model
data = pd.read_parquet(ROOT / "data" / "processed" / f"model_{MODE}.parquet")

groups = data["project"].astype(str)
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=CFG["random_seed"])
train_idx, temp_idx = next(gss1.split(data, groups=groups))

train = data.iloc[train_idx].copy()
temp = data.iloc[temp_idx].copy()

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=CFG["random_seed"] + 1)
val_rel, test_rel = next(gss2.split(temp, groups=temp["project"].astype(str)))

val = temp.iloc[val_rel].copy()
test = temp.iloc[test_rel].copy()

print("Train projects:", sorted(train["project"].unique()))
print("Validation projects:", sorted(val["project"].unique()))
print("Test projects:", sorted(test["project"].unique()))

assert set(train["project"]).isdisjoint(set(val["project"]))
assert set(train["project"]).isdisjoint(set(test["project"]))
assert set(val["project"]).isdisjoint(set(test["project"]))

Train projects: ['FLEX', 'HIVE', 'JRACLOUD', 'JRASERVER', 'MC', 'MCPE', 'MDEV', 'SERVER']
Validation projects: ['CONFSERVER', 'SAK']
Test projects: ['OSSRH', 'QTBUG']


In [3]:
from src.features import feature_spec, sample_training_rows
from src.models import fit_cox, save_artifact
from src.evaluation import evaluate_survival_model

strict = bool(CFG["features"]["strict_no_leakage"])
landmark = int(CFG["features"]["landmark_days"])
numeric_cols, categorical_cols = feature_spec(
    MODE,
    strict_no_leakage=strict,
    landmark_days=landmark,
)

max_rows = CFG["models"].get("max_train_rows")
train_fit = sample_training_rows(
    train,
    max_rows=max_rows,
    random_state=CFG["random_seed"],
)

print("Train rows used:", len(train_fit), "/", len(train))
print("Numerical:", numeric_cols)
print("Categorical:", categorical_cols)

Train rows used: 100000 / 576250
Numerical: ['created_weekday', 'created_hour', 'initial_assigned']
Categorical: ['initial_priority', 'initial_issue_type']


In [4]:
cox_artifact = fit_cox(
    train_df=train_fit,
    numeric_cols=numeric_cols,
    categorical_cols=categorical_cols,
    alpha=float(CFG["models"]["cox"]["alpha"]),
)

metrics_val = evaluate_survival_model(
    cox_artifact,
    train_df=train_fit,
    test_df=val,
    horizons_days=CFG["evaluation"]["horizons_days"],
)
metrics_test = evaluate_survival_model(
    cox_artifact,
    train_df=train_fit,
    test_df=test,
    horizons_days=CFG["evaluation"]["horizons_days"],
)

pd.DataFrame([
    {"split": "validation", **metrics_val},
    {"split": "test", **metrics_test},
])

,split,n_test,events_test,harrell_c,ipcw_c,auc_30,auc_60,auc_90,mean_dynamic_auc,ibs
0,validation,87261,76376,0.595495,0.588819,0.635460,0.640160,0.643017,0.636529,0.198562
1,test,172132,149713,0.374037,0.378573,0.309602,0.338403,0.352223,0.312770,NaN


In [5]:
model_dir = ROOT / "results" / "models"
table_dir = ROOT / "results" / "tables"
model_dir.mkdir(parents=True, exist_ok=True)
table_dir.mkdir(parents=True, exist_ok=True)

save_artifact(cox_artifact, model_dir / f"cox_{MODE}.joblib")

metrics_df = pd.DataFrame([
    {"model": "CoxPH", "mode": MODE, "split": "validation", **metrics_val},
    {"model": "CoxPH", "mode": MODE, "split": "test", **metrics_test},
])
metrics_df.to_csv(table_dir / f"cox_metrics_{MODE}.csv", index=False)

split_df = data[["repository", "project", "issue_key"]].copy()
split_df["split"] = "unused"
split_df.loc[train.index, "split"] = "train"
split_df.loc[val.index, "split"] = "validation"
split_df.loc[test.index, "split"] = "test"
split_df.to_csv(table_dir / f"split_{MODE}.csv", index=False)

print("Saved model + metrics + split.")

Saved model + metrics + split.


In [6]:
# Hazard ratios
coef = np.asarray(cox_artifact["model"].coef_, dtype=float)
feature_names = cox_artifact["feature_names"]

hr = pd.DataFrame({
    "feature": feature_names,
    "coef": coef,
    "hazard_ratio": np.exp(coef),
})
hr["abs_log_hr"] = np.abs(np.log(hr["hazard_ratio"]))

hr = hr.sort_values("abs_log_hr", ascending=False)
hr.to_csv(table_dir / f"cox_hazard_ratios_{MODE}.csv", index=False)

hr.head(30)

,feature,coef,hazard_ratio,abs_log_hr
32,cat__initial_issue_type_Public Security Vulner...,1.429042,4.174699,1.429042
38,cat__initial_issue_type_Suggestion,-1.361973,0.256155,1.361973
36,cat__initial_issue_type_Story Task,-1.285151,0.276609,1.285151
39,cat__initial_issue_type_Support Request,1.274288,3.576155,1.274288
34,cat__initial_issue_type_Request,-1.208462,0.298656,1.208462
44,cat__initial_issue_type_Wish,-0.932504,0.393567,0.932504
33,cat__initial_issue_type_Question,0.726877,2.068611,0.726877
22,cat__initial_issue_type_Build Failure,0.695400,2.004512,0.695400
19,cat__initial_priority_Unknown,0.684451,1.982683,0.684451
43,cat__initial_issue_type_Third-party issue,0.620568,1.859983,0.620568


## PH assumption

Cox PH có giả định proportional hazards.  
Với dữ liệu lớn, hãy chạy kiểm tra Schoenfeld/lifelines trên **mẫu hợp lý** và báo cáo biến nào vi phạm.  
RSF ở notebook 07 không phụ thuộc giả định PH này.

Notebook này ưu tiên tạo baseline/model trước; kiểm tra PH là bước bắt buộc khi viết kết quả cuối.